# PandasDataFrameOutputParser → "조회 계획"을 구조화 출력으로 받고, 실행은 코드로

`PandasDataFrameOutputParser`는 LangChain v1에서 **`langchain-classic`** 으로 이동한 레거시 API입니다. 또한 책에서 권장한 `gpt-3.5-turbo`는 OpenAI API에서 **2026-10-23 종료 예정**입니다.

이 파서는 LLM이 `column:Age`, `mean:Fare[0..4]` 같은 **자체 미니 언어**로 답하게 한 뒤 파서가 이를 해석하는 방식이었습니다. 모델이 이 문법을 틀리기 쉬웠던 이유입니다(책의 "잘못 형식화된 쿼리" 예시).

현재 권장되는 설계는 다음과 같습니다.

1. **LLM은 무엇을 조회할지(계획)만** 구조화 출력으로 결정한다. → 컬럼명은 `Literal`(enum)로 제한되어 존재하지 않는 컬럼을 고를 수 없음
2. **실제 계산은 우리 코드가** pandas로 수행한다. → 결과가 결정적이고, 임의 코드 실행 위험이 없음

> 참고: `langchain_experimental`의 `create_pandas_dataframe_agent`처럼 LLM이 생성한 파이썬 코드를 그대로 실행하는 방식도 있지만, 임의 코드 실행 위험 때문에 `allow_dangerous_code=True`를 명시해야 하며 샌드박스 환경에서만 사용하는 것이 권장됩니다.

In [ ]:
# 최초 1회 설치 (LangChain v1 기준)
# %pip install -qU langchain langchain-openai langchain-classic python-dotenv pandas

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 의 OPENAI_API_KEY, LANGSMITH_API_KEY 를 불러옵니다.

# LangSmith 추적: 별도 헬퍼 없이 환경변수만 설정하면 자동으로 활성화됩니다.
if os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "CH03-OutputParser")

In [ ]:
from langchain.chat_models import init_chat_model

# 공급자 중립적인 모델 초기화 ("공급자:모델명")
# 다른 모델로 바꾸려면 문자열만 교체하면 됩니다. 예) "anthropic:claude-sonnet-4-5", "ollama:llama3.1"
llm = init_chat_model("openai:gpt-4.1-mini", temperature=0)

`titanic.csv` 데이터를 로드합니다.

In [ ]:
import pandas as pd

df = pd.read_csv("./data/titanic.csv")
df.head()

## 1. 조회 계획 스키마 정의

- `ColumnName`은 **실제 DataFrame의 컬럼 목록**으로 만든 `Literal`입니다. JSON Schema의 `enum`으로 변환되어 모델은 이 중에서만 고를 수 있습니다.
- `X | None` 타입이지만 기본값을 주지 않았으므로 "필수이지만 null 허용" 필드가 됩니다(OpenAI strict 모드 요구사항과 호환).

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field

ColumnName = Literal[tuple(df.columns)]  # 예: Literal["PassengerId", "Survived", ...]


class DataFrameQuery(BaseModel):
    """사용자 요청을 DataFrame 조회 작업으로 변환한 결과"""

    operation: Literal["column", "row", "mean", "sum", "min", "max", "count"] = Field(
        description="column: 컬럼 값 조회, row: 행 조회, 그 외: 해당 컬럼의 집계"
    )
    column: ColumnName | None = Field(description="대상 컬럼. row 조회이면 null")
    row_start: int | None = Field(description="시작 행 번호(0부터 시작, 포함). 전체이면 null")
    row_end: int | None = Field(description="끝 행 번호(포함). 전체이면 null")

## 2. 계획을 실행하는 함수 (계산은 pandas가 수행)

In [ ]:
def run_query(df: pd.DataFrame, q: DataFrameQuery):
    start = q.row_start if q.row_start is not None else 0
    stop = q.row_end + 1 if q.row_end is not None else None
    subset = df.iloc[start:stop]

    if q.operation == "row":
        return subset

    if q.column is None:
        raise ValueError(f"'{q.operation}' 작업에는 column 이 필요합니다.")
    series = subset[q.column]

    if q.operation == "column":
        return series
    if q.operation == "count":
        return int(series.count())
    if not pd.api.types.is_numeric_dtype(series):
        raise ValueError(f"'{q.column}' 컬럼은 숫자형이 아니라서 {q.operation} 을(를) 계산할 수 없습니다.")
    return getattr(series, q.operation)()

## 3. 체인 구성

프롬프트에 컬럼과 dtype 정보를 넣어 모델이 적절한 컬럼을 고르도록 돕습니다.
`plan_chain`은 계획만, `query_chain`은 계획 + 실행까지 수행합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "사용자의 요청을 DataFrame 조회 계획으로 변환하세요.\n"
            "DataFrame 컬럼과 타입:\n{schema}",
        ),
        ("human", "{question}"),
    ]
).partial(schema=df.dtypes.to_string())

plan_chain = prompt | llm.with_structured_output(DataFrameQuery)
query_chain = plan_chain | RunnableLambda(lambda q: run_query(df, q))

### 컬럼 조회

In [ ]:
plan = plan_chain.invoke({"question": "Age column 을 조회해 주세요."})
print(plan)

query_chain.invoke({"question": "Age column 을 조회해 주세요."}).head()

### 첫 번째 행 조회

In [ ]:
print(plan_chain.invoke({"question": "Retrieve the first row."}))
query_chain.invoke({"question": "Retrieve the first row."})

### 일부 행의 평균 (0~4행 Age 평균)

In [ ]:
result = query_chain.invoke({"question": "Retrieve the average of the Ages from row 0 to 4."})
print("LLM 계획 기반 결과:", result)
print("직접 계산 검증   :", df["Age"].head().mean())

### 전체 Fare 평균

책에서는 이 질문이 "잘못 형식화된 쿼리"의 예시였습니다. 계획을 구조화 출력으로 받으면 동일한 질문도 안정적으로 처리됩니다.

In [ ]:
result = query_chain.invoke({"question": "Calculate average `Fare` rate."})
print("LLM 계획 기반 결과:", result)
print("직접 계산 검증   :", df["Fare"].mean())

## (참고) 레거시 API

기존 코드 유지보수가 목적이라면 `from langchain_classic.output_parsers import PandasDataFrameOutputParser`로 import할 수 있습니다. 신규 코드에는 권장하지 않습니다.